## atemptation 

In [1]:
print("kernel ok")

kernel ok


In [2]:
import os
import findspark

findspark.init('/home/ubuntu/spark')

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Jupyter-Interactive-Analysis") \
    .master("spark://g23-master:7077") \
    .getOrCreate()

print("Spark is successfully connected")
spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/09 20:32:41 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark is successfully connected


In [ ]:

df = spark.read.parquet("/data/cleaned_reddit/")
#all column names and data types.
df.printSchema()


root
 |-- author: string (nullable = true)
 |-- body: string (nullable = true)
 |-- content: string (nullable = true)
 |-- content_len: long (nullable = true)
 |-- id: string (nullable = true)
 |-- normalizedBody: string (nullable = true)
 |-- subreddit: string (nullable = true)
 |-- subreddit_id: string (nullable = true)
 |-- summary: string (nullable = true)
 |-- summary_len: long (nullable = true)
 |-- title: string (nullable = true)



In [ ]:
#First 3 rows of the data, with full content
df.show(3, truncate=False)

+-----------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [3]:
import pyspark.sql.functions as F

print("Reading data from HDFS...")
df = spark.read.parquet("/data/cleaned_reddit/")

print("Calculating Top 10 hottest subreddits...")
top_subreddits = df.groupBy("subreddit") \
                   .count() \
                   .orderBy(F.col("count").desc()) \
                   .limit(10)

top_subreddits.show()

Reading data from HDFS...


Calculating Top 10 hottest subreddits...


+-------------------+------+
|          subreddit| count|
+-------------------+------+
|          AskReddit|589947|
|      relationships|352049|
|    leagueoflegends|109307|
|               tifu| 52219|
|relationship_advice| 50416|
|              trees| 47286|
|             gaming| 43851|
|            atheism| 43268|
|      AdviceAnimals| 40783|
|              funny| 40171|
+-------------------+------+



In [4]:
import pyspark.sql.functions as F

# Find the top 10 most active authors, excluding deleted accounts
print("Calculating Top 10 Most Active Authors...")

active_authors = df.filter(df.author != "[deleted]") \
                   .groupBy("author") \
                   .count() \
                   .orderBy(F.col("count").desc()) \
                   .limit(10)

active_authors.show()

Calculating Top 10 Most Active Authors...


+---------------+-----+
|         author|count|
+---------------+-----+
|   iamtotalcrap| 3108|
|RamsesThePigeon|  889|
|        DejaBoo|  829|
|     Shaper_pmp|  649|
|        codayus|  621|
|    Death_Star_|  577|
|        rand486|  573|
|         kuvter|  549|
| HeathenCyclist|  480|
|         Lots42|  454|
+---------------+-----+



In [5]:
print("Keyword Trend - Subreddits with the most 'Help/Advice' requests...")

# Define target keywords using regex (case-insensitive)
# Looking for phrases like "need help", "please help", "need advice"
keyword_regex = "(?i)(need help|please help|need advice)"

# Filter posts containing the keywords and aggregate by subreddit
help_requests = df.filter(F.col("body").rlike(keyword_regex)) \
    .groupBy("subreddit") \
    .count() \
    .withColumnRenamed("count", "help_post_count") \
    .orderBy(F.col("help_post_count").desc()) \
    .limit(10)

help_requests.show()

Keyword Trend - Subreddits with the most 'Help/Advice' requests...


+-------------------+---------------+
|          subreddit|help_post_count|
+-------------------+---------------+
|      relationships|          26402|
|          AskReddit|           4819|
|relationship_advice|           3999|
|                sex|           1191|
|             Advice|           1053|
|    leagueoflegends|            866|
|    TwoXChromosomes|            775|
|         depression|            742|
|              trees|            728|
|      dating_advice|            725|
+-------------------+---------------+



In [6]:
import pyspark.sql.functions as F

print("Distributed Word Count on Summaries...")

# Convert text to lowercase, split by non-alphabetic characters, and explode into multiple rows
words_df = df.select(
    F.explode(F.split(F.lower(F.col("summary")), "[^a-zA-Z]+")).alias("word")
)

# Filter out empty strings and short meaningless words (length <= 4)
filtered_words_df = words_df.filter(F.length(F.col("word")) > 4)

# Trigger a massive cluster shuffle to group and count the words
top_words = filtered_words_df.groupBy("word") \
    .count() \
    .orderBy(F.col("count").desc()) \
    .limit(15)

top_words.show()

Distributed Word Count on Summaries...


+-------+------+
|   word| count|
+-------+------+
|  about|383624|
| people|272779|
|because|263104|
| should|238105|
|  there|233049|
|  would|227440|
|  think|188757|
| really|178994|
|  being|176049|
|  other|172211|
|  their|153298|
|  still|143712|
|  after|138701|
| friend|138632|
| better|131858|
+-------+------+



##League

In [7]:
import pyspark.sql.functions as F

print("Task 1: Analyzing 'Nerf' vs 'Buff' discussion lengths in League of Legends...")

# Filter ONLY League of Legends posts
lol_df = df.filter(F.col("subreddit") == "leagueoflegends")

# Create boolean columns checking if the post body contains 'nerf' or 'buff'
balance_df = lol_df.withColumn("mentions_nerf", F.col("body").rlike("(?i)\\bnerf\\b|\\bnerfs\\b")) \
                   .withColumn("mentions_buff", F.col("body").rlike("(?i)\\bbuff\\b|\\bbuffs\\b"))

# Aggregate to see which category writes longer posts on average
balance_stats = balance_df.groupBy("mentions_nerf", "mentions_buff") \
                          .agg(
                              F.count("*").alias("total_posts"),
                              F.avg("content_len").cast("int").alias("avg_post_length")
                          ) \
                          .orderBy(F.col("total_posts").desc())

balance_stats.show()

Task 1: Analyzing 'Nerf' vs 'Buff' discussion lengths in League of Legends...


+-------------+-------------+-----------+---------------+
|mentions_nerf|mentions_buff|total_posts|avg_post_length|
+-------------+-------------+-----------+---------------+
|        false|        false|      97847|            196|
|        false|         true|       6064|            356|
|         true|        false|       3400|            296|
|         true|         true|       1996|            447|
+-------------+-------------+-----------+---------------+



In [8]:
print("Task 2: Categorizing posts into 'Esports' vs 'Casual Play'...")

# Define keywords for Esports and Casual/Ranked play
esports_keywords = "(?i)\\b(lcs|lec|lck|lpl|worlds|msi|faker|t1|fnatic|g2)\\b"
casual_keywords = "(?i)\\b(soloq|elo|ranked|lp|smurf|climbing)\\b"

# Label each post based on its content
categorized_lol_df = lol_df.withColumn(
    "discussion_type",
    F.when(F.col("body").rlike(esports_keywords) & F.col("body").rlike(casual_keywords), "Both")
     .when(F.col("body").rlike(esports_keywords), "Esports / Pro Play")
     .when(F.col("body").rlike(casual_keywords), "Ranked / Casual Play")
     .otherwise("General / Other")
)

# Aggregate the results
category_counts = categorized_lol_df.groupBy("discussion_type") \
                                    .count() \
                                    .orderBy(F.col("count").desc())

category_counts.show(truncate=False)

Task 2: Categorizing posts into 'Esports' vs 'Casual Play'...


+--------------------+-----+
|discussion_type     |count|
+--------------------+-----+
|General / Other     |76320|
|Ranked / Casual Play|24720|
|Esports / Pro Play  |6921 |
|Both                |1346 |
+--------------------+-----+

